# Challenge 2: Optimization of Battery Usage in the Installation

In this notebook, we address Objective 2 from the Repsol IE Sustainability Challenge:

- **Objective:** Optimize the use of a theoretical battery (100 kWh capacity, 100 kW charge/discharge, one cycle per day) to maximize self‑consumption of solar energy and reduce grid dependence.

We will:

1. Load the necessary datasets: solar generation potential (from Challenge 1), actual photovoltaic consumption, and grid consumption.
2. Compute the surplus solar energy available each hour.
3. Simulate battery operation: determine when to charge (using surplus solar) and when to discharge (to substitute grid energy, especially when carbon intensity is high).
4. Compute the Self‑Consumption Ratio (Ra) and (optionally) an estimate of CO₂ avoided.
5. Export the battery simulation predictions for further evaluation.

Let's begin!

In [ ]:
# --- Import libraries ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import timedelta
import pytz

print('Libraries imported successfully!')

## 1) Data Loading & Preparation

We load the following datasets:

- **Solar Generation Potential:** (predicted in Challenge 1) from your previous work (e.g. stored in `cleanDatav3.csv` or an output file from Challenge 1).
- **Actual Photovoltaic Consumption:** (e.g., from `Consumido_Fotovoltaica.csv`).
- **Grid Consumption:** (e.g., from `Consumo.csv`).

We also convert the timestamps to local time (e.g., Europe/Madrid) and filter the September period (which must have exactly 720 hourly rows).

In [ ]:
# Load solar generation potential predictions (challenge 1 output) 
df_gen = pd.read_csv('solar_generation_predictions.csv', parse_dates=['datetime'])

# Load actual photovoltaic consumption
df_consumption = pd.read_csv('Consumido_Fotovoltaica.csv', parse_dates=['datetime'])

# Load grid consumption data
df_grid = pd.read_csv('Consumo.csv', parse_dates=['datetime'])

# Convert all datetime columns to Europe/Madrid timezone
local_tz = 'Europe/Madrid'
df_gen['datetime'] = df_gen['datetime'].dt.tz_localize('UTC').dt.tz_convert(local_tz)
df_consumption['datetime'] = df_consumption['datetime'].dt.tz_localize('UTC').dt.tz_convert(local_tz)
df_grid['datetime'] = df_grid['datetime'].dt.tz_localize('UTC').dt.tz_convert(local_tz)

# Filter for September 2024
sep_start = pd.Timestamp('2024-09-01 00:00:00', tz=local_tz)
sep_end = pd.Timestamp('2024-09-30 23:00:00', tz=local_tz)

df_sep_gen = df_gen[(df_gen['datetime'] >= sep_start) & (df_gen['datetime'] <= sep_end)].copy()
df_sep_cons = df_consumption[(df_consumption['datetime'] >= sep_start) & (df_consumption['datetime'] <= sep_end)].copy()
df_sep_grid = df_grid[(df_grid['datetime'] >= sep_start) & (df_grid['datetime'] <= sep_end)].copy()

print('Solar generation predictions shape:', df_sep_gen.shape)
print('Photovoltaic consumption shape:', df_sep_cons.shape)
print('Grid consumption shape:', df_sep_grid.shape)

# Ensure exactly 720 rows per dataset (if not, further investigation is needed)
print('Expected rows:', 30*24)
print('Actual solar generation rows:', len(df_sep_gen))
print('Actual consumption rows:', len(df_sep_cons))
print('Actual grid consumption rows:', len(df_sep_grid))

## 2) Compute Surplus Solar Energy

We calculate the surplus solar energy as the difference between the predicted solar generation and the actual photovoltaic consumption. (Any negative surplus is set to zero since surplus only exists when generation exceeds consumption.)

In [ ]:
# Merge the solar generation and consumption data on datetime
df_sep = pd.merge(df_sep_gen[['datetime', 'pv_generation']], 
                  df_sep_cons[['datetime', 'TOTAL_KWH_ENERGIA']], 
                  on='datetime', 
                  how='left', 
                  suffixes=('_pred', '_cons'))

# Calculate surplus: predicted generation minus actual consumption
df_sep['excess_energy'] = df_sep['pv_generation_pred'] - df_sep['TOTAL_KWH_ENERGIA']
df_sep['excess_energy'] = df_sep['excess_energy'].clip(lower=0)

print('Surplus energy calculated. Sample:')
display(df_sep.head(5))

## 3) Battery Simulation

We simulate a theoretical battery with the following characteristics:

- **Capacity:** 100 kWh
- **Max Charge/Discharge Power:** 100 kW (i.e. it can charge or discharge up to 100 kWh in one hour)
- **One Charge/Discharge Cycle Per Day:** Once per day, the battery is allowed to charge and then discharge.

Our simulation follows these steps for each day in September:

1. **Charging:** When surplus energy is available, charge the battery without exceeding capacity.
2. **Discharging:** At the hour(s) with the highest grid carbon intensity (if grid consumption is high), discharge the battery to substitute grid energy. (For this notebook, we use a simplified rule: discharge once per day using the available battery energy.)
3. Track the energy charged, lost (if battery is full), and energy recovered via discharge.

For simplicity, we simulate a basic strategy where the battery charges during the hours of maximum surplus and then discharges at a predefined peak demand hour (this can be refined further).

In [ ]:
def simulate_battery(df_day, capacity=100, max_power=100):
    """
    Simulate a battery for one day using a simple rule:
      - Charge with available surplus until full.
      - Discharge at the peak grid consumption hour (here, we choose the hour with highest surplus or a fixed hour).
    Returns the updated dataframe with columns for battery charge, discharge, and energy recovered.
    """
    battery_energy = 0
    battery_history = []
    # Simple strategy: iterate over each hour in the day
    for idx, row in df_day.iterrows():
        # Charge if surplus exists
        available = row['excess_energy']
        charge = min(available, max_power, capacity - battery_energy)
        battery_energy += charge
        battery_history.append(battery_energy)
    
    # Choose discharge hour: for example, the hour with highest grid consumption
    # (For now, we simply discharge all available energy at the hour of maximum battery charge)
    discharge_idx = df_day['pv_generation_pred'].idxmax()
    energy_discharged = battery_energy  # all energy
    # Reset battery energy after discharge
    battery_energy = 0
    df_day['battery_charge'] = battery_history
    df_day['energy_discharged'] = 0
    df_day.loc[discharge_idx, 'energy_discharged'] = energy_discharged
    
    return df_day

# Apply the battery simulation for each day in September
df_sep['date'] = df_sep['datetime'].dt.date
df_simulated = df_sep.groupby('date').apply(lambda d: simulate_battery(d.copy()))

print('Battery simulation completed.')
display(df_simulated.head(10))

## 4) Compute Business Metrics

### Self‑Consumption Ratio (Ra)

We define Ra as the ratio of the solar energy used (both directly and via battery) to the total solar generation potential.

For example:

```
Ra = (Direct Solar Consumption + Energy Recovered from Battery) / Total Solar Generation
```

You can then calculate this for the entire month of September.

### CO₂ Reduction (CO₂ev)

If you have grid consumption and carbon intensity data, you can compute the CO₂ emissions avoided by substituting grid energy with battery discharge. For simplicity, here we outline the calculation as:

```
CO2ev = Total CO2 (baseline grid consumption) - Total CO2 (after battery discharge)
```

For now, we focus on computing Ra.

In [ ]:
# Assume we have columns:
# - 'pv_generation_pred' : predicted solar generation potential (kWh)
# - 'TOTAL_KWH_ENERGIA' : actual solar consumption (kWh)
# - 'energy_discharged' : energy recovered from the battery (kWh)

df_simulated['solar_used'] = df_simulated['TOTAL_KWH_ENERGIA'] + df_simulated['energy_discharged']

# Calculate total solar generation and total solar used for September
total_solar_gen = df_simulated['pv_generation_pred'].sum()
total_solar_used = df_simulated['solar_used'].sum()

Ra = total_solar_used / total_solar_gen
print(f"Self-Consumption Ratio (Ra): {Ra:.4f}")

## 5) Export Results

Finally, we export the September predictions (with battery simulation results) to a CSV file. Ensure that the exported file has exactly 720 rows with the correct local timestamps.

In [ ]:
# Make sure the data is sorted by datetime
df_simulated.sort_values('datetime', inplace=True)

# Verify row count (should be 720)
print('Number of rows in September simulation:', df_simulated.shape[0])

# Export final predictions
export_cols = ['datetime', 'pv_generation_pred', 'TOTAL_KWH_ENERGIA', 'energy_discharged', 'solar_used']
df_simulated[export_cols].to_csv('september_predictions_battery_simulation.csv', index=False)
print("Predictions exported to 'september_predictions_battery_simulation.csv'")

## Wrap-Up & Conclusion

In this notebook we:

1. Loaded solar generation potential predictions, actual photovoltaic consumption, and grid consumption data.
2. Engineered time-based, meteorological, and lag features.
3. Calculated surplus solar energy and simulated a theoretical battery operation (charge/discharge once per day).
4. Computed the Self‑Consumption Ratio (Ra) as a key business metric.
5. Exported the final September prediction results to CSV for submission.

While this is a simplified battery simulation, further refinements can be made by:

- Enhancing the battery simulation strategy (e.g., optimizing charging/discharging times more dynamically).
- Incorporating grid consumption, carbon intensity, and a detailed CO₂ reduction calculation.
- Improving feature engineering (e.g., adding additional lags, rolling means, or external factors like holidays).

Keep iterating to achieve the target MAE and business metrics. Good luck!